In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import random
import pickle

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 64          # T
batch_size = 128
max_iters = 3000
learning_rate = 3e-4
eval_iters = 100        # reports the loss
eval_interval = 500
n_embd = 384            # no of features
n_head = 8              # no of heads run in parallel
n_layer = 8             # no of decoder blocks
dropout = 0.2

cuda


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)

In [3]:
# Character level Tokenizer

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

# encode("hello")
# decode([66, 63, 70, 70, 73])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72])


In [4]:
## data splitting between training and validation

n = int(0.8*len(data))
train_data = data[:n].to(device)
val_data = data[n:].to(device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, 1))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i+1: i + block_size + 1] for i in ix])
    # x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch(train_data)
print(x.shape)
print(y)

torch.Size([128, 64])
tensor([[73, 72,  1,  ..., 66,  1, 78],
        [76, 61, 66,  ..., 81,  1, 78],
        [62,  1, 67,  ..., 63, 72,  1],
        ...,
        [64,  1, 78,  ..., 67, 60, 79],
        [ 1, 77, 63,  ..., 63, 72,  8],
        [ 1, 78, 66,  ..., 69, 59,  8]], device='cuda:0')


In [5]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Input: B, T, channels
        # Output: B, T, hs
        B, T, C = x.shape
        k = self.key(x)    # B, T, hs
        q = self.query(x)  # B, T, hs

        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        # No of features each head captures
        head_size = n_embd // n_head
        # Self Attention
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y) # Add and Norm
        y = self.ffwd(x)
        x = self.ln2(x + y) # Add and Norm
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        #no of decoder blocks/layers we have running sequentially
        # * unpacks the list into separate elements, turning them into arguments
        self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])

        # Layer Norm 
        self.ln_f = nn.LayerNorm(n_embd)
        # language modelling head 
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

    def forward(self, index, targets = None):
        B, T = index.shape

       # index and targets are both (B, T) tensor of integers
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device = device))
        x = tok_emb + pos_emb 
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
model = GPTLanguageModel(vocab_size)
m = model.to(device)
# print(f"Model Parameters: {sum(p.numel() for p in m.parameters()):, }")


In [6]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# Optimizer with cosine LR schedule
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr(it):
    warmup_iters = 50  # shorter warmup for fewer iterations
    min_lr = learning_rate / 10
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

for iter in range(max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    if iter % eval_iters == 0:
        losses = estimate_loss() 
        print(f"step {iter:4d} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    
    optimizer.zero_grad(set_to_non`e=True)
    loss.backward()
    # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(loss.item())

step    0 | train: 2.3745 | val: 2.5825
step  100 | train: 2.0553 | val: 2.3149
step  200 | train: 1.7699 | val: 2.1056
step  300 | train: 1.5959 | val: 1.9793
step  400 | train: 1.4738 | val: 1.9176
step  500 | train: 1.3921 | val: 1.8602
step  600 | train: 1.3220 | val: 1.8185
step  700 | train: 1.2623 | val: 1.7994
step  800 | train: 1.2116 | val: 1.7697
step  900 | train: 1.1665 | val: 1.7645
step 1000 | train: 1.1273 | val: 1.7583
step 1100 | train: 1.0863 | val: 1.7623
step 1200 | train: 1.0534 | val: 1.7530
step 1300 | train: 1.0153 | val: 1.7567
step 1400 | train: 0.9858 | val: 1.7795
step 1500 | train: 0.9546 | val: 1.7831
step 1600 | train: 0.9237 | val: 1.7968
step 1700 | train: 0.8942 | val: 1.7932
step 1800 | train: 0.8741 | val: 1.8163
step 1900 | train: 0.8478 | val: 1.8397
step 2000 | train: 0.8242 | val: 1.8408
step 2100 | train: 0.8012 | val: 1.8559
step 2200 | train: 0.7842 | val: 1.8656
step 2300 | train: 0.7697 | val: 1.8708
step 2400 | train: 0.7523 | val: 1.8786
